# Hugging Face Diffusion Course — Unit 1: Introduction to Diffusers

Source: https://huggingface.co/learn/diffusion-course/unit1/1

This notebook starts the Unit 1 work with a small scaffold only. The goal is to keep the first pass close to the official course before adding runnable cells.

# Introduction to 🤗 Diffusers

This notebook follows Hugging Face Diffusion Course Unit 1:

https://huggingface.co/learn/diffusion-course/unit1/1

We will use the original Colab notebook as a guide, but run the work locally in this project.
The goal is to understand the main Diffusers pieces before training a small butterfly image generator.

## What this notebook will build toward

In the original notebook, we eventually:

- try a ready-made diffusion pipeline
- load image data from the Hugging Face Hub
- add noise with a scheduler
- train a small UNet model
- assemble the pieces into a mini image-generation pipeline

For now, we will go step by step and inspect each part before moving on.

## Step 1: Setup check

The Colab notebook installs packages with `%pip install`.
In this local repo, we use `uv` and keep dependencies in `pyproject.toml`.

Before importing anything, confirm that the environment can see the packages we need.

In [2]:
import time
from pathlib import Path

start = time.perf_counter()

import accelerate
import diffusers
import torch
import transformers


elapsed = time.perf_counter() - start
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
output_dir = Path("../outputs/hf-course") if Path.cwd().name == "notebooks" else Path("outputs/hf-course")
output_dir.mkdir(parents=True, exist_ok=True)

print(f"setup imports: {elapsed:.2f}s")
print("diffusers:", diffusers.__version__)
print("transformers:", transformers.__version__)
print("accelerate:", accelerate.__version__)
print("torch:", torch.__version__)
print("device:", device)
print("output dir:", output_dir.resolve())

setup imports: 3.57s
diffusers: 0.38.0
transformers: 5.12.0
accelerate: 1.14.0
torch: 2.11.0+cu128
device: cuda
output dir: C:\Users\giloz\dev\visual-genai-lab\outputs\hf-course


In [1]:
import datasets

In [ ]:
from PIL import Image


def make_grid(images: list[Image.Image], size: int = 64) -> Image.Image:
    """Place PIL images next to each other for quick visual inspection.

    Args:
        images: Generated images to arrange in one row.
        size: Width and height for each resized image tile.

    Returns:
        A single PIL image containing all input images in a row.
    """
    output_image = Image.new("RGB", (size * len(images), size))

    for index, image in enumerate(images):
        output_image.paste(image.resize((size, size)), (index * size, 0))

    return output_image